# PS14 AI Cyber Defence — XGBoost Detection Engine (Reference Notebook)

> **NOTE**: This notebook is a reference example. The production pipeline runs via `main.py`.
> Do not use hardcoded paths or dataset-specific field names here.
> All configuration is driven by `config/config.yaml`.

## Usage

Supply your telemetry CSV path in the cell below. The notebook demonstrates:
- Loading and cleaning raw telemetry
- Fitting the FeatureEncoder
- Training the XGBoostDetector
- Inspecting predicted attack probabilities

**Model quality is NOT evaluated here.** Evaluation will be performed
separately after training on the actual operational telemetry dataset.

In [ ]:
import sys
import os
import pandas as pd
import numpy as np
import yaml

# Ensure project root is on the path when running from notebooks/
sys.path.append(os.path.abspath(".."))

from src.ingestion import TelemetryLoader
from src.preprocessing import TelemetryCleaner
from src.features import FeatureEncoder
from src.detection import XGBoostDetector

# Load configuration
with open("../config/config.yaml", "r") as f:
    config = yaml.safe_load(f)

print("Loaded configuration:", config['pipeline']['name'], 'v' + config['pipeline']['version'])

In [ ]:
# ── Supply your telemetry data path here ──────────────────────────────────────
# Replace this with the actual path to your telemetry file.
# Supported formats: CSV, JSON, JSONL, Parquet.
TELEMETRY_PATH = None  # e.g. "../data/raw/my_telemetry.csv"

if TELEMETRY_PATH is None:
    raise ValueError(
        "Set TELEMETRY_PATH to your telemetry file above.\n"
        "The notebook does not generate or fabricate telemetry."
    )

loader = TelemetryLoader()
raw_df = loader.load(TELEMETRY_PATH)
print(f"Loaded {len(raw_df):,} rows × {len(raw_df.columns)} columns.")
raw_df.head(3)

In [ ]:
# Preprocess and Encode Features
pp_cfg = config['preprocessing']
label_cfg = config['label']

cleaner = TelemetryCleaner(
    drop_columns=pp_cfg.get('drop_columns', []),
    timestamp_col=pp_cfg.get('timestamp_col', ''),
    max_nan_fraction=pp_cfg.get('max_nan_fraction', 0.5),
    numeric_impute_strategy=pp_cfg.get('numeric_impute_strategy', 'median'),
)
cleaned_df = cleaner.clean(raw_df)

encoder = FeatureEncoder(
    target_col=pp_cfg['target_col'],
    benign_values=label_cfg.get('benign_values', []),
    categorical_columns=pp_cfg.get('categorical_columns', []),
    hex_columns=pp_cfg.get('hex_columns', []),
    labels_already_binary=label_cfg.get('labels_already_binary', False),
)

X, y = encoder.fit_transform(cleaned_df)
print(f"Feature matrix: {X.shape}")
print(f"Label distribution:\n{y.value_counts().to_dict()}")

In [ ]:
# Train XGBoost Detection Engine
model_cfg = config['model']
detector = XGBoostDetector(
    **model_cfg['xgboost'],
    use_smote=model_cfg.get('use_smote', True),
    train_test_split_ratio=model_cfg.get('train_test_split', 0.2),
)
X_val, y_val, val_probs = detector.train(X, y)

# Inspect attack probabilities on the full feature matrix
attack_probs = detector.predict_attack_probability(X)
print("Sample attack_probability values (first 10):", attack_probs[:10].round(4))
print(f"Attack prob range: [{attack_probs.min():.4f}, {attack_probs.max():.4f}]")

In [ ]:
# Save model artifacts for use in predict mode
import os

paths = config['paths']
sup_dir = os.path.join('..', paths['supervised_model_dir'])
os.makedirs(sup_dir, exist_ok=True)

encoder.save(os.path.join(sup_dir, 'preprocessor.pkl'))
detector.save_model(os.path.join(sup_dir, 'xgboost_model.pkl'))

print("Artifacts saved.")
print("  preprocessor.pkl")
print("  xgboost_model.pkl")
print()
print("Feature importances (top 10):")
print(detector.get_feature_importances().head(10))